In [0]:
%sql
GRANT USE CATALOG ON CATALOG `fk_ce1366` TO `fk_analyst`;

GRANT USE SCHEMA, SELECT ON SCHEMA `fk_ce1366`.gold TO `fk_analyst`;

In [0]:
%sql
GRANT USE CATALOG ON CATALOG `fk_ce1366` TO `fk_engineers`;

GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA `fk_ce1366`.bronze TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA `fk_ce1366`.silver TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA `fk_ce1366`.gold TO `fk_engineers`;
GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA `fk_ce1366`.ops TO `fk_engineers`;

GRANT READ VOLUME, WRITE VOLUME ON VOLUME `fk_ce1366`.`landing`.`raw` TO `fk_engineers`;

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS fk_ce1366.ops.user_entitlements (
  user_email    STRING,
  allowed_city  STRING,
  can_view_pii  BOOLEAN
);
""")

spark.sql("""
INSERT INTO fk_ce1366.ops.user_entitlements
VALUES
  ('alice@example.com', 'ALL',          true),
  ('bob@example.com',   'San Francisco', false),
  ('carol@example.com', 'New York',      false);
""")

spark.table("fk_ce1366.ops.user_entitlements").display()

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION fk_ce1366.ops.mask_email(val STRING)
RETURNS STRING
RETURN CASE
  WHEN val IS NULL THEN NULL
  WHEN is_member('fk_admins')
    OR EXISTS (
      SELECT 1
      FROM fk_ce1366.ops.user_entitlements
      WHERE user_email = current_user()
        AND can_view_pii = TRUE
    )
  THEN val
  ELSE CONCAT(
         SUBSTRING(SPLIT(val, '@')[0], 1, 1),
         '***@',
         SPLIT(val, '@')[1]
       )
END
""")

spark.sql("""
CREATE OR REPLACE FUNCTION fk_ce1366.ops.mask_phone(val STRING)
RETURNS STRING
RETURN CASE
  WHEN val IS NULL THEN NULL
  WHEN is_member('fk_admins')
    OR EXISTS (
      SELECT 1
      FROM fk_ce1366.ops.user_entitlements
      WHERE user_email = current_user()
        AND can_view_pii = TRUE
    )
  THEN val
  ELSE CONCAT('XXXXXX', RIGHT(val, 4))
END
""")


spark.sql("""
ALTER TABLE fk_ce1366.gold.dim_customer
ALTER COLUMN email SET MASK fk_ce1366.ops.mask_email
""")

spark.sql("""
ALTER TABLE fk_ce1366.gold.dim_customer
ALTER COLUMN phone SET MASK fk_ce1366.ops.mask_phone
""")

spark.sql("""
SELECT column_name, mask_name
FROM information_schema.columns
WHERE table_catalog = 'fk_ce1366'
  AND table_schema  = 'gold'
  AND table_name    = 'dim_customer'
  AND column_name IN ('email', 'phone')
""").display()